# Notebook 07 — Evaluate T790M Mutant + Cross-Target Comparison

Day 9 goals:
1. Repeat evaluation pipeline for 4I22 molecules
2. Cross-target chemical space comparison (UMAP)
3. Compare score distributions WT vs T790M


In [ ]:
import os, sys, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src import MolEvaluator, evaluate_batch, VinaDocker, dock_molecules, load_config
from src.utils import sdf_to_smiles_list, merge_vina_and_metrics, filter_top_candidates
import pandas as pd

cfg = load_config(f'{PROJECT_ROOT}/configs/targets.yaml')
samp_cfg = load_config(f'{PROJECT_ROOT}/configs/sampling.yaml')

SDF_FILE    = f'{PROJECT_ROOT}/results/generated/4I22_raw.sdf'
VINA_CSV    = f'{PROJECT_ROOT}/results/vina_scores/4I22_vina.csv'
METRICS_CSV = f'{PROJECT_ROOT}/results/vina_scores/4I22_metrics.csv'

In [ ]:
# ── Metrics ─────────────────────────────────────────────────────────────────
smiles_list = sdf_to_smiles_list(SDF_FILE)
metrics_df = evaluate_batch(smiles_list, output_csv=METRICS_CSV)

In [ ]:
# ── Vina docking on 4I22 ────────────────────────────────────────────────────
with open(f'{PROJECT_ROOT}/data/pocket_centers.json') as f:
    centers = json.load(f)

docker = VinaDocker(
    receptor_pdbqt=f'{PROJECT_ROOT}/data/pdb/4I22_receptor.pdbqt',
    center=tuple(centers['4I22']),
    exhaustiveness=samp_cfg['evaluation']['vina_exhaustiveness'],
)
vina_df = dock_molecules(SDF_FILE, docker, VINA_CSV)

In [ ]:
# ── Cross-target UMAP chemical space ────────────────────────────────────────
from src.visualization import plot_chemical_space

wt_smiles  = [s for s in sdf_to_smiles_list(f'{PROJECT_ROOT}/results/generated/1M17_raw.sdf') if s]
mut_smiles = [s for s in smiles_list if s]

baseline_smiles = {name: info['smiles'] for name, info in cfg['baselines'].items()}

plot_chemical_space(
    wt_smiles[:500],   # subsample for speed
    mut_smiles[:500],
    baseline_smiles,
    output_path=f'{PROJECT_ROOT}/results/figures/fig5_umap.png',
)

In [ ]:
# ── Score distribution comparison ───────────────────────────────────────────
from src.visualization import plot_score_distribution

wt_vina_df  = pd.read_csv(f'{PROJECT_ROOT}/results/vina_scores/1M17_vina.csv')
mut_vina_df = pd.read_csv(VINA_CSV)
baseline_scores = {
    row['drug']: row['vina_score']
    for _, row in pd.read_csv(f'{PROJECT_ROOT}/results/vina_scores/baselines.csv').iterrows()
    if row['target'] == '1M17'
}

plot_score_distribution(
    wt_vina_df, mut_vina_df, baseline_scores,
    output_path=f'{PROJECT_ROOT}/results/figures/fig2_vina_distribution.png',
)